In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report

C:\Users\Ertuğrul\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
data = {
    "reviewText": [
        "Bu restoran harika, yemekler çok lezzetli!",
        "Hizmet mükemmeldi, kesinlikle tavsiye ederim.",
        "Çok kötü bir deneyimdi, bir daha asla gitmem.",
        "Garsonlar çok ilgili ve ortam çok güzel.",
        "Yemeklerin tadı berbattı, hiç beğenmedim.",
        "Müşteri hizmetleri gerçekten çok iyiydi.",
        "Sipariş çok geç geldi ve soğuktu.",
        "Kesinlikle harika bir mekan, tekrar geleceğim.",
        "Menü çok sınırlıydı ve fiyatlar yüksekti.",
        "Lezzetli tatlılar ve hızlı servis, çok memnun kaldım!",
        "Konser harikaydı, sanatçılar mükemmeldi.",
        "Ürün beklediğim gibi kaliteli ve güzel paketlenmişti.",
        "Tadı çok güzeldi, bayıldım!",
        "Film kesinlikle izlemeye değer, çok sürükleyiciydi.",
        "Hizmet kalitesi gerçekten çok iyi, kesinlikle tavsiye ederim.",
        "Fiyatına göre çok iyi performans gösteren bir ürün.",
        "Kahve çok tazeydi, aroması harikaydı.",
        "Paketleme çok özenliydi, hiçbir zarar görmeden elime ulaştı.",
        "Müşteri desteği çok hızlı yanıt verdi ve sorunumu çözdü.",
        "Deneyimim çok güzeldi, herkese öneririm.",
        "Bu otel inanılmazdı, manzarası harikaydı.",
        "Eğlenceli ve öğretici bir deneyim yaşadım.",
        "Çalışanlar çok güler yüzlü ve yardımseverdi.",
        "Bu ürün beklentilerimi fazlasıyla karşıladı.",
        "Kullanımı çok kolay ve verimli bir ürün.",
        "Film çok sıkıcıydı, hiç beğenmedim.",
        "Ürün kırık geldi, çok kalitesizdi.",
        "Müşteri hizmetleri çok ilgisizdi, hiç yardımcı olmadılar.",
        "Sipariş çok geç geldi, memnun kalmadım.",
        "Tat hiç beklediğim gibi değildi, hayal kırıklığı yaşadım.",
        "Ürün açıklamadaki gibi çıkmadı, geri iade edeceğim.",
        "Fiyatına göre kesinlikle değmez, çok kötüydü.",
        "Restoran çok kalabalıktı ve servis çok yavaştı.",
        "Tatlı çok bayattı, hiç beğenmedim.",
        "Konser çok kötüydü, ses sistemi berbattı.",
        "Paketleme çok özensizdi, ürün zarar görmüştü.",
        "Film hiç akıcı değildi, zaman kaybıydı.",
        "Müşteri hizmetleri bana yanlış bilgi verdi.",
        "Ürün kalitesi çok kötü, param boşa gitti.",
        "Hizmet kalitesi berbattı, tekrar asla tercih etmem.",
        "Sipariş ettiklerim eksik geldi, çok sinir bozucuydu.",
        "Kahve çok bayattı, tadı kötüydü.",
        "Teslimat çok geç yapıldı, hiç memnun kalmadım.",
        "Ürün açıklamalardaki gibi değildi, tamamen aldatıcı.",
        "Mekan çok gürültülüydü, rahat oturamadım.",
        "Satıcı hiç yardımcı olmadı, ilgisizdi.",
        "Fiyatı aşırı pahalı ve kalitesizdi.",
        "Yemekler soğuktu, hiç lezzetli değildi.",
        "Ürün elime ulaştığında hasarlıydı.",
        "Burası kesinlikle bir daha gitmeyeceğim bir yer."
    ],
    "label": [
        "pozitif", "pozitif", "negatif", "pozitif", "negatif",
        "pozitif", "negatif", "pozitif", "negatif", "pozitif",
        "pozitif", "pozitif", "pozitif", "pozitif", "pozitif",
        "pozitif", "pozitif", "pozitif", "pozitif", "pozitif",
        "pozitif", "pozitif", "pozitif", "pozitif", "pozitif",
        "negatif", "negatif", "negatif", "negatif", "negatif",
        "negatif", "negatif", "negatif", "negatif", "negatif",
        "negatif", "negatif", "negatif", "negatif", "negatif",
        "negatif", "negatif", "negatif", "negatif", "negatif",
        "negatif", "negatif", "negatif", "negatif", "negatif"
    ]
}

df = pd.DataFrame(data)

In [3]:
#Normalizing Case Folding

df["reviewText"]=df["reviewText"].str.lower()

#Punctuations
df["reviewText"]=df["reviewText"].str.replace("[^\w\s]"," ",regex=True)

#Numbers
df["reviewText"]=df["reviewText"].str.replace("\d"," ",regex=True)


In [4]:
import nltk

sw=stopwords.words("turkish")
df["reviewText"]=df["reviewText"].apply(lambda x: " ".join(x for x in str(x).split() if x not in sw))


In [5]:
#Tokenization
#nltk.download("punkt")
from textblob import TextBlob
df["reviewText"].apply(lambda x: TextBlob(x).words).head()

0               [restoran, harika, yemekler, lezzetli]
1    [hizmet, mükemmeldi, kesinlikle, tavsiye, ederim]
2            [kötü, bir, deneyimdi, bir, asla, gitmem]
3                    [garsonlar, ilgili, ortam, güzel]
4             [yemeklerin, tadı, berbattı, beğenmedim]
Name: reviewText, dtype: object

In [6]:
from sklearn.utils import shuffle

# Veriyi karıştırma
df = shuffle(df, random_state=42)

df.head()

,reviewText,label
13,film kesinlikle izlemeye değer sürükleyiciydi,pozitif
39,hizmet kalitesi berbattı tekrar asla tercih etmem,negatif
30,ürün açıklamadaki çıkmadı geri iade edeceğim,negatif
45,satıcı yardımcı olmadı ilgisizdi,negatif
17,paketleme özenliydi hiçbir zarar görmeden elim...,pozitif


In [7]:
X_train, X_test, y_train, y_test = train_test_split(df["reviewText"], df["label"], test_size=0.2, random_state=42)

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 1))

# Eğitim verisini dönüştürme
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Test verisini dönüştürme
X_test_tfidf = tfidf_vectorizer.transform(X_test)

In [9]:
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

nb = MultinomialNB()
cv_scores = cross_val_score(nb, X_train_tfidf, y_train, cv=5, scoring='accuracy')
print(f"Cross-validation doğruluğu: {cv_scores.mean():.2f}")


Cross-validation doğruluğu: 0.60


In [10]:
param_grid = {'alpha': [0.1, 0.5, 1.0, 1.5]}  # Naive Bayes için hiperparametre aralığı
grid_search = GridSearchCV(MultinomialNB(), param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_train_tfidf, y_train)

GridSearchCV(cv=5, estimator=MultinomialNB(),
             param_grid={'alpha': [0.1, 0.5, 1.0, 1.5]}, scoring='accuracy')

In [11]:
print(f"En iyi parametreler: {grid_search.best_params_}")
print(f"En iyi cross-validation doğruluğu: {grid_search.best_score_}")

best_model = grid_search.best_estimator_



En iyi parametreler: {'alpha': 0.1}
En iyi cross-validation doğruluğu: 0.6


In [12]:

best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_tfidf)
test_accuracy = accuracy_score(y_test, y_pred)
print(f"Test setindeki doğruluk: {test_accuracy:.2f}")

Test setindeki doğruluk: 0.60
